# Redução de Dados em Python

A redução de dados é um processo essencial para o tratamento das imagens tiradas em astrofotografia, sendo o processo de otimiza-las e tornar os valores captados mais próximos da realidade.

#### Captação

Ao captar fótons em uma CCD, elétrons são gerados via efeito fotoelétrico, essas são as fontes de interesse quando trabalhamos com astrofotografia, porém, nem todos os elétrons gerados pelo conversor são fontes de interesse.

## Imagens de Calibração

Com isso, surgem as imagens de calibração, que visam corrigir efeitos relacionados a fontes fora do campo de estudo e os próprios defeitos de fabricação, tanto do telescópio, quanto da câmera.

#### Flat

O Flat é retirado utilizando uma base branca iluminada de forma homogênea, seu objetivo é mapear a sensibilidade dos pixels associados a CCD, de forma a corrigir qualquer irregularidade vinda da captação direta de fótons. É possível fazer Sky Flats, no anoitecer ou amanhecer, quando o céu é iluminado de forma aproximadamente homogênea.

Corrige problemas relacionados a objetos no caminho entre a fonte e a CCD.

#### Dark

O Dark é retirado com o equipamento ótico tampado, seu objetivo é captar o ruído proveniente dos elétrons gerados por outro meios, que não o fotoelétrico. Normalmente, é possível captar ruído térmico e eventuais raios cósmicos.

#### Bias

o bias é utilizado para caracterizar e corrigir o nível de sinal de fundo introduzido pela eletrônica da CCD, sendo imagens tiradas com o tempo de exposição o mais próximo possível de zero, o equipamento
dita o valor mínimo possível.

### Import das Bibliotecas necessária

Neste código, usaremos as bibliotecas:

- os - Para identificar diretórios e arquivos
- Numpy - Para calculos relacionados a matrizes de dados
- Astropy - Para a leitura de arquivos .fits e extração dos dados
- matplotlib - Para a geração de imagens

In [ ]:
import os
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

### Diretórios

O programa trabalha com os seguintes diretórios, para melhor organização:

In [ ]:
dirbias = "bias"
dirdark = "dark"
dirflat = "flat"
dirlight = "light"

dirmasters = "masters"
dirfinal = "final"

### Definindo Funções Auxiliares

- listarfits - Gera uma lista organizada em ordem alfabetica com todos os arquivos .fits
- leiturafits - Extrai os dados de cada pixel e as informações dos metadados da imagem
- imcombine - Gera uma pilha com determinadas imagens e retira a mediana da pilha, gerando uma imagem master
- salvar - Salva as imagens na pasta "masters"
- criarmaster - Chama todas as funções anteriores de forma a gerar e salvar as imagens master

In [ ]:
def listarfits(dir):
    if not os.path.isdir(dir):
        return []
    
    imagens = []
    print(f'Lendo imagens em {dir}')

    for img in os.listdir(dir):
        if os.path.isfile(os.path.join(dir, img)):
            if img.endswith(".fits"):
                imagens.append(os.path.join(dir, img))

    if len(imagens) == 0:
        print(f"{dir} vazia, pulando...")
    
    return sorted(imagens)

def leiturafits(img):
    with fits.open(img) as hdul:
        dados = hdul[0].data 
        info = hdul[0].header
    
    return dados, info

def imcombine(img):
    print(f'Criando master com imagens...')
    if len(img) == 0:
        return None, None

    imagens = []

    for arq in img:
        print(arq)
        dados, info = leiturafits(arq)
        imagens.append(dados)

    y, x = imagens[0].shape
    pilha = np.zeros((len(imagens), y, x))
    for i in range(len(imagens)):
        pilha[i] = imagens[i]

    mediana = np.median(pilha, axis=0)

    return mediana, info

def salvar(dir, dados, info):
    fits.writeto(dir, dados, info, overwrite=True)
    print(f"{dir} salvo em masters")

def criarmaster(dir):
    lista = listarfits(dir)
    master, info = imcombine(lista)

    nome = os.path.basename(dir)
    nome = os.path.splitext(nome)[0]
    salvar(os.path.join(dirmasters, f"master_{nome}.fits"), master, info)

### Criando Masters

Cada imagem será representada por uma matriz de NxN pixels, onde N é a resolução da imagem, cada elemento da matriz será o valor atribuído a 1 pixel.

O parametro t_light e t_dark devem ser alterados com o objetivo de calibrar as imagens em relação ao tempo de exposição dos lights

Aqui chamamos a função que cria a imagem master, logo em seguida, extraímos os dados dos masters e plotamos a imagem resultante:

In [ ]:
criarmaster(dirbias)
criarmaster(dirflat)
darkcru = listarfits(dirdark)

os.makedirs("darkdiv", exist_ok=True)

t_light = 35
t_dark = 1200

for img in darkcru:
    dadod, infod = leiturafits(img)
    darkdiv = (t_light / t_dark) * dadod

    nome = os.path.basename(img)
    nome = os.path.splitext(nome)[0]
    fits.writeto(os.path.join("darkdiv", f"{nome}_darkdiv.fits"), darkdiv, infod, overwrite=True)

Mbias, infobias = leiturafits(os.path.join(dirmasters, "master_bias.fits"))
Mdark, infodark = leiturafits(os.path.join(dirmasters, "master_dark.fits"))
Mflat, infoflat = leiturafits(os.path.join(dirmasters, "master_flat.fits"))

plt.imshow(Mbias, cmap='gray')
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title('Master Bias')
plt.show()
print("Min:", np.min(Mbias))
print("Max:", np.max(Mbias))
print("Mediana:", np.median(Mbias))

plt.imshow(Mdark, cmap='gray', vmin=2998, vmax=3000)
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title('Master Dark')
plt.show()
print("Min:", np.min(Mdark))
print("Max:", np.max(Mdark))
print("Mediana:", np.median(Mdark))

plt.imshow(Mflat, cmap='gray')
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title('Master Flat')
plt.show()
print("Min:", np.min(Mflat))
print("Max:", np.max(Mflat))
print("Mediana:", np.median(Mflat))

### Calibração do Master Flat

Aqui fazemos uma subtração entre as matrizes de dados de cada imagem master,

In [ ]:
print("Calibrando master_flat")
flatCal = Mflat - Mbias

plt.imshow(flatCal, cmap='gray', vmin=19000, vmax=22000)
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title('Master Flat - Master Bias')
plt.show()
print("Min:", np.min(flatCal))
print("Max:", np.max(flatCal))
print("Mediana:", np.median(flatCal))

In [ ]:
print("Normalizando master_flat")
medflat = np.median(Mflat)
flatNorm = flatCal/medflat

fits.writeto(os.path.join(dirmasters, "master_flat_norm.fits"), flatNorm, infoflat, overwrite=True)
print("master_flat normalizado.")

plt.imshow(flatNorm, cmap='gray', vmin=0.9, vmax=1)
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title('Master Flat Normalizado')
plt.show()
print("Min:", np.min(flatNorm))
print("Max:", np.max(flatNorm))
print("Mediana:", np.median(flatNorm))

In [ ]:
for img in listarfits(dirlight):
    print(f'Reduzindo {img}')
    light, infolight = leiturafits(img)
    lightCal = light-Mbias-Mdark
    lightRed = lightCal/flatNorm

    nome = os.path.basename(img)
    nome = os.path.splitext(nome)[0]
    fits.writeto(os.path.join(dirfinal, f"{nome}_red.fits"), lightRed, infolight, overwrite=True)

print("Tudo pronto")

In [ ]:
data1, info = leiturafits(os.path.join(dirlight, "HW_Vir_0196.fits"))
datasat = 50*data1

plt.imshow(data1, cmap='gray', vmin=0, vmax=5000)
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title("HW_Vir_0196.fits")
plt.show()
print("Min:", np.min(data1))
print("Max:", np.max(data1))
print("Mediana:", np.median(data1))

data2, info = leiturafits(os.path.join(dirdark, "DARK_1503.fits"))

plt.imshow(data2, cmap='gray')
plt.colorbar()
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.title("HW_Vir_0196_red.fits")
plt.show()
print("Min:", np.min(data2))
print("Max:", np.max(data2))
print("Mediana:", np.median(data2))